# Inserting, Updating and Deleting Data

In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [2]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [4]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_store")

In [6]:
pd.read_sql("show tables", con=engine)

,Tables_in_sql_store
0,customers
1,order_item_notes
2,order_items
3,order_statuses
4,orders
5,products
6,shippers


## Inserting a row
this is the structure of `customers` table in sql_store databaseL:

![](images\1.png)

In [14]:
query = sqlalchemy.text("""
insert into customers
values(
	default,
    "Arman",
    "Salehi",
    default,
    null, 
    "address",
    "city",
    "CA",
    default
)
""")


with engine.begin() as conn:
    conn.execute(query)

as a best practice it's better to select what columns that you want to insert values into(optionally you can change the order of columns):

In [18]:
query = sqlalchemy.text("""
insert into customers(
    first_name,
    last_name,
    birth_date,
    address,
    city,
    state)

values(
    "Arman",
    "Salehi",
    "2003-09-21",
    "address",
    "city",
    "CA")
""")


with engine.begin() as conn:
    conn.execute(query)

## Inserting Multiple Rows

In [19]:
query = sqlalchemy.text("""
insert into shippers(name)
values ("shipper1"),
	   ("shipper2"),
       ("shipper3")
""")


with engine.begin() as conn:
    conn.execute(query)

## Inserting Hierarchical Rows
this code creates an order in orders table and then attaches two items to that exact order in one go in the order_items table:

In [23]:
query = sqlalchemy.text("""
insert into orders(customer_id, order_date, status)
values (1, "2026-01-01", 1);
insert into order_items
values (last_insert_id(), 1, 1, 2.95),
	   (last_insert_id(), 2, 1, 3.95);
""")


with engine.begin() as conn:
    conn.execute(sqlalchemy.text("""insert into orders(customer_id, order_date, status)
    values (1, "2026-01-01", 1)"""))

    conn.execute(sqlalchemy.text("""insert into order_items
    values (last_insert_id(), 1, 1, 2.95), (last_insert_id(), 2, 1, 3.95)"""))

## Creating a Copy of a Table

In [35]:
query = sqlalchemy.text("""
create table invoces_archived as
select *
from sql_invoicing.invoices i
join sql_invoicing.clients c
	using(client_id)
where payment_date is not null
""")


with engine.begin() as conn:
    conn.execute(query)

# NOTICE: BECAUSE WE ARE CONNECTED TO THE 'sql_store' DATABASE, 'invoices_archived' TABLE IS CREATED IN THIS PATH

## Updating a Single Row

In [37]:
query = sqlalchemy.text("""
update sql_invoicing.invoices
set payment_total = 10, payment_date = "2026-01-01"
where invoice_id = 1
""")

with engine.begin() as conn:
    conn.execute(query)

## Updating Multiple Rows

In [38]:
 query = sqlalchemy.text("""
update customers 
set points = points + 50
where birth_date <= '1990-01-01'
""")

with engine.begin() as conn:
    conn.execute(query)

## Updating Multiple Rows

In [40]:
 query = sqlalchemy.text("""
update customers 
set points = points + 50
where birth_date <= '1990-01-01'
""")

with engine.begin() as conn:
    conn.execute(query)

## Using Subqueries in Updates

In [41]:
 query = sqlalchemy.text("""
update orders
set comments = "gold customer"
where customer_id in
			(select customer_id
			from customers
			where points > 3000)
""")

with engine.begin() as conn:
    conn.execute(query)

## Deleting Rows

In [42]:
 query = sqlalchemy.text("""
delete from customers
where customers.first_name = "Arman"
""")

with engine.begin() as conn:
    conn.execute(query)

In [45]:
 query = sqlalchemy.text("""
delete from order_items
where order_id in (11, 12, 13)
""")

with engine.begin() as conn:
    conn.execute(query)

In [47]:
 query = sqlalchemy.text("""
delete from orders
where order_id in (11, 12, 13)
""")

with engine.begin() as conn:
    conn.execute(query)

In [48]:
query = sqlalchemy.text("""
delete from shippers 
where name like "shipper%"
""")

with engine.begin() as conn:
    conn.execute(query)